<!--
Licensed to the Apache Software Foundation (ASF) under one or more
contributor license agreements.  See the NOTICE file distributed with
this work for additional information regarding copyright ownership.
The ASF licenses this file to You under the Apache License, Version 2.0
(the "License"); you may not use this file except in compliance with
the License.  You may obtain a copy of the License at

   http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# 07 · E7 — Coverage as a metric

Validate that coverage % (intrinsic, cheap) tracks the graded score (extrinsic,
expensive). If it does, future evals can use coverage instead of the full sweep.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
import eval_common as ec
import eval_v2 as v2
import seagate_scoring as scoring

FIXTURE = 'seagate_multi'
fdir = v2.fixture_dir(FIXTURE)
manifest = v2.load_table_manifest(FIXTURE)

# Multi-schema project: primary schema seagate_ops + the seagate_core scope.
# seagate_ref is deliberately EXCLUDED (pulling it in is an E9 failure).
config = ec.EvalConfig.from_env(
    schema_name='seagate_ops',
    results_dir=fdir.parent.parent / 'evaluation' / 'results' / 'seagate_multi',
)
client = v2.AgentClientV2(config, schema_names=['seagate_core', 'seagate_ops'])
me = client.login()
print('Authenticated as:', me.get('username') or me)

# Preconditions: Postgres required (R4); memory loop must be off (R1, warn-only).
pre = client.assert_eval_preconditions(require_postgres=True)
print('DB backend:', pre['backend'])
for w in pre['warnings']:
    print('WARNING:', w)

questions = ec.parse_test_queries(fdir / 'test_queries.md')
print('Loaded', len(questions), 'graded questions (Q1-Q18)')
glossary = (fdir / 'bi_glossary.md').read_text(encoding='utf-8')

Authenticated as: admin


DB backend: postgresql
Loaded 18 graded questions (Q1-Q18)


In [2]:
def score_sweep(results):
    """Score a result set with the exact ground-truth scorer (incl. Q16-Q18)."""
    verdicts = {r['id']: scoring.score_result(
        r['id'], r['result_rows'], r['answer_summary']) for r in results}
    correct = sum(1 for v in verdicts.values() if v in scoring.CORRECT_VERDICTS)
    return correct, verdicts

### Capture (coverage, graded score) for wren_base and wren_bi

In [3]:
print('archived', client.clean_baseline(), 'project(s)')
project = client.resolve_project(); pid = project['id']
client.onboard(pid)
points = []

# wren_base
base_cov = client.wait_for_coverage(pid)
base_correct, _ = score_sweep(ec.run_experiment(client, 'wren_base', questions, save=False))
points.append(('wren_base', base_cov, base_correct))

# wren_bi
doc = client.create_document_from_text(pid, glossary, 'bi_glossary.md')
signal = client.enrich_round(pid, doc['id'])
bi_correct, _ = score_sweep(ec.run_experiment(client, 'wren_bi', questions, save=False))
points.append(('wren_bi', signal['coverage'], bi_correct))

import pandas as pd
pts = pd.DataFrame(points, columns=['condition', 'coverage', 'graded_correct'])
print(pts.to_string(index=False))

archived 1 project(s)


condition  coverage  graded_correct
wren_base       NaN               4
  wren_bi     0.417               5


### Correlation + offline deterministic cross-check

The live coverage score is LLM-judged (non-deterministic). For a rigorous bound,
hand-label each glossary claim (covered/partial/missing) per condition as gold and
score the detector with the deterministic `coverage_eval.score_coverage` — that
removes judge noise. The gold labels are a TODO fixture (EVAL_V2_SPEC.md E7).

In [4]:
# Spearman across the (coverage, score) points once you have ≥3 conditions/seeds.
if len(pts) >= 3:
    print('Spearman:', pts['coverage'].corr(pts['graded_correct'], method='spearman'))
else:
    print('Add more conditions/seeds for a meaningful correlation.')
print('\nCoverage rises base -> bi:',
      pts.iloc[0]['coverage'], '->', pts.iloc[1]['coverage'])

Add more conditions/seeds for a meaningful correlation.

Coverage rises base -> bi: nan -> 0.417
